In [14]:
import pandas as pd

df = pd.read_csv('data/Restaurant reviews.csv')

print("Original shape:", df.shape)
print("Columns:", df.columns.tolist())
print("Missing values:\n", df.isna().sum())
print("Exact duplicate rows:", df.duplicated().sum())
display(df.head(3))

Original shape: (10000, 8)
Columns: ['Restaurant', 'Reviewer', 'Review', 'Rating', 'Metadata', 'Time', 'Pictures', '7514']
Missing values:
 Restaurant       0
Reviewer        38
Review          45
Rating          38
Metadata        38
Time            38
Pictures         0
7514          9999
dtype: int64
Exact duplicate rows: 36


,Restaurant,Reviewer,Review,Rating,Metadata,Time,Pictures,7514
0,Beyond Flavours,Rusha Chakraborty,"The ambience was good, food was quite good . h...",5,"1 Review , 2 Followers",5/25/2019 15:54,0,2447.0
1,Beyond Flavours,Anusha Tirumalaneedi,Ambience is too good for a pleasant evening. S...,5,"3 Reviews , 2 Followers",5/25/2019 14:20,0,NaN
2,Beyond Flavours,Ashok Shekhawat,A must try.. great food great ambience. Thnx f...,5,"2 Reviews , 3 Followers",5/24/2019 22:54,0,NaN


In [15]:
import re

# Standardize column names and remove the nearly empty extra column.
df.columns = df.columns.str.strip().str.lower()
df = df.drop(columns=["7514"], errors="ignore")

# Remove exact duplicates before changing values.
before_duplicates = len(df)
df = df.drop_duplicates().copy()
print("Duplicate rows removed:", before_duplicates - len(df))

# Normalize text fields; keep missing values as missing.
for col in ["restaurant", "reviewer", "review", "rating", "metadata", "time"]:
    df[col] = df[col].astype("string").str.replace(r"\s+", " ", regex=True).str.strip()
    df[col] = df[col].replace("", pd.NA)

# Data types.
df["rating"] = pd.to_numeric(df["rating"], errors="coerce")
df["time"] = pd.to_datetime(df["time"], format="%m/%d/%Y %H:%M", errors="coerce")
df["pictures"] = pd.to_numeric(df["pictures"], errors="coerce")

# Reviewer profile metadata, e.g. '3 Reviews , 2 Followers'.
meta = df["metadata"].fillna("")
df["reported_reviews"] = pd.to_numeric(
    meta.str.extract(r"(\d+)\s+Reviews?\b", flags=re.IGNORECASE, expand=False),
    errors="coerce"
)
df["followers"] = pd.to_numeric(
    meta.str.extract(r"(\d+)\s+Followers?\b", flags=re.IGNORECASE, expand=False),
    errors="coerce"
)
df["followers_not_listed"] = df["followers"].isna()

# Review length in words; missing written reviews have length 0.
df["review_length"] = df["review"].fillna("").str.split().str.len()

print("Rows before filtering:", len(df))
print("Invalid/missing ratings:", df["rating"].isna().sum())
print("Unparsed review-count metadata:", df["reported_reviews"].isna().sum())
print("Missing written review text:", df["review"].isna().sum())
display(df[["reviewer", "rating", "reported_reviews", "followers", "review_length"]].head())

Duplicate rows removed: 36
Rows before filtering: 9964
Invalid/missing ratings: 3
Unparsed review-count metadata: 8
Missing written review text: 9


,reviewer,rating,reported_reviews,followers,review_length
0,Rusha Chakraborty,5.0,1,2,41
1,Anusha Tirumalaneedi,5.0,3,2,27
2,Ashok Shekhawat,5.0,2,3,31
3,Swapnil Sarkar,5.0,1,1,28
4,Dileep,5.0,3,2,27


In [16]:
required = ["restaurant", "reviewer", "rating", "time", "pictures", "reported_reviews"]
before_filter = len(df)
clean = df.dropna(subset=required).copy()

# No followers is treated as zero.
clean["followers"] = clean["followers"].fillna(0).astype(int)
clean["reported_reviews"] = clean["reported_reviews"].astype(int)
clean["pictures"] = clean["pictures"].astype(int)
clean["review_length"] = clean["review_length"].astype(int)

clean = clean[[
    "restaurant", "reviewer", "review", "rating", "time",
    "pictures", "review_length", "reported_reviews", "followers", "followers_not_listed"
]]

print("Rows removed during filtering:", before_filter - len(clean))
print("Clean reviews:", len(clean))
print("Distinct reviewer names:", clean["reviewer"].nunique())
print("Rating-only rows (no written text):", clean["review"].isna().sum())
print("Rows where follower count wasn't listed:", clean["followers_not_listed"].sum())
display(clean.head())

Rows removed during filtering: 9
Clean reviews: 9955
Distinct reviewer names: 7442
Rating-only rows (no written text): 7
Rows where follower count wasn't listed: 1578


,restaurant,reviewer,review,rating,time,pictures,review_length,reported_reviews,followers,followers_not_listed
0,Beyond Flavours,Rusha Chakraborty,"The ambience was good, food was quite good . h...",5.0,2019-05-25 15:54:00,0,41,1,2,False
1,Beyond Flavours,Anusha Tirumalaneedi,Ambience is too good for a pleasant evening. S...,5.0,2019-05-25 14:20:00,0,27,3,2,False
2,Beyond Flavours,Ashok Shekhawat,A must try.. great food great ambience. Thnx f...,5.0,2019-05-24 22:54:00,0,31,2,3,False
3,Beyond Flavours,Swapnil Sarkar,Soumen das and Arun was a great guy. Only beca...,5.0,2019-05-24 22:11:00,0,28,1,1,False
4,Beyond Flavours,Dileep,Food is good.we ordered Kodi drumsticks and ba...,5.0,2019-05-24 21:37:00,0,27,3,2,False


In [17]:
reviewer_df = clean.groupby("reviewer", as_index=False).agg(
    reviews_in_dataset=("rating", "size"),
    reported_reviews=("reported_reviews", "max"),
    followers=("followers", "max"),
    avg_review_length=("review_length", "mean"),
    avg_pictures=("pictures", "mean"),
    avg_rating=("rating", "mean"),
)

print("Reviewer feature matrix shape:", reviewer_df.shape)
display(reviewer_df.head())
print("Reviewers represented by only one review:", (reviewer_df["reviews_in_dataset"] == 1).sum())
print("Missing values in reviewer features:\n", reviewer_df.isna().sum())

Reviewer feature matrix shape: (7442, 7)


,reviewer,reviews_in_dataset,reported_reviews,followers,avg_review_length,avg_pictures,avg_rating
0,#FOODPORN,1,35,5873,26.0,0.0,5.0
1,#chloesviews // CAH,1,3,1,48.0,0.0,5.0
2,$ign,1,50,55,82.0,2.0,4.0
3,4mybanti,1,7,4,34.0,0.0,2.0
4,@FuloriBinaChutneyKaiseBani,1,86,244,80.0,0.0,5.0


Reviewers represented by only one review: 6103
Missing values in reviewer features:
 reviewer              0
reviews_in_dataset    0
reported_reviews      0
followers             0
avg_review_length     0
avg_pictures          0
avg_rating            0
dtype: int64


In [18]:
clean.to_csv("data/restaurant_reviews_cleaned.csv", index=False, lineterminator="\n")
reviewer_df.to_csv("data/reviewer_features.csv", index=False, lineterminator="\n")

print("Saved data/restaurant_reviews_cleaned.csv")
print("Saved data/reviewer_features.csv")

Saved restaurant_reviews_cleaned.csv
Saved reviewer_features.csv
